# Quick XGBoost Baseline for Loan Payback Prediction
# Playground Series - Season 5, Episode 11

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [2]:
train = pd.read_csv('/kaggle/input/playground-series-s5e11/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e11/test.csv')

In [3]:
train.shape

(593994, 13)

In [4]:
test.shape

(254569, 12)

In [5]:
train['loan_paid_back'].value_counts(normalize=True)

loan_paid_back
1.0    0.79882
0.0    0.20118
Name: proportion, dtype: float64

## featrues Prepare

In [6]:
X = train.drop(['id', 'loan_paid_back'], axis=1)
y = train['loan_paid_back']
X_test = test.drop('id', axis=1)

In [7]:
categorical_cols = ['gender', 'marital_status', 'education_level','employment_status', 'loan_purpose', 'grade_subgrade']

## Label Encode

In [8]:
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Fit on combined train and test to handle all categories
    combined = pd.concat([X[col], X_test[col]])
    le.fit(combined)
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])
    label_encoders[col] = le

## Data Split

In [9]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [10]:
X_train.shape

(475195, 11)

In [11]:
X_val.shape

(118799, 11)

## model Training

In [12]:
model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    n_jobs=-1
)

In [13]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=100,
    verbose=100
)

[0]	validation_0-auc:0.90850
[100]	validation_0-auc:0.91868
[200]	validation_0-auc:0.91977
[300]	validation_0-auc:0.92038
[400]	validation_0-auc:0.92047
[460]	validation_0-auc:0.92045


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=-1,
              num_parallel_tree=None, random_state=42, ...)

## Validataion performance

In [14]:
val_pred = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_pred)
print(f"\nValidation AUC: {val_auc:.4f}")


Validation AUC: 0.9205


## retrian on Full train data

In [15]:
model_full = xgb.XGBClassifier(
    n_estimators=model.best_iteration,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

In [16]:
model_full.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=361, n_jobs=-1,
              num_parallel_tree=None, random_state=42, ...)

## Prediction on test data

In [17]:
test_pred = model_full.predict_proba(X_test)[:, 1]

## Submission file

In [18]:
submission = pd.DataFrame({
    'id': test['id'],
    'loan_paid_back': test_pred
})

In [19]:
submission.to_csv('submission.csv', index=False)

In [20]:
submission.shape

(254569, 2)

In [21]:
submission.head()

,id,loan_paid_back
0,593994,0.926365
1,593995,0.978992
2,593996,0.374025
3,593997,0.909653
4,593998,0.972422
